# Recommendation Systems Walkthrough

Build recommendation systems using real movie data - the same techniques Netflix, Spotify, and Amazon use!

**Topics:**
1. Loading MovieLens Data
2. Collaborative Filtering (User-Based CF, SVD)
3. Content-Based Filtering
4. Evaluation
5. Cold Start Problem

## Setup

In [1]:
# Uncomment and run ONCE if you don't have Surprise installed
# (You only need to do this one time, then you can skip this cell)

# import sys
# !{sys.executable} -m pip install scikit-surprise

In [2]:
import pandas as pd
import numpy as np
from collections import defaultdict
from sklearn.metrics.pairwise import cosine_similarity
from surprise import Dataset, Reader, SVD, KNNBasic
from surprise.model_selection import train_test_split as surprise_split
from surprise import accuracy

import warnings
warnings.filterwarnings('ignore')
print("Libraries loaded!")

Libraries loaded!


---
## Part 1: Loading MovieLens Data

The **MovieLens 100K** dataset contains 100,000 movie ratings from real users.

In [4]:
# Load MovieLens data for exploration
data_path = Dataset.load_builtin('ml-100k', prompt=False)

# Load as DataFrames for exploration
ratings_url = 'https://files.grouplens.org/datasets/movielens/ml-100k/u.data'
movies_url = 'https://files.grouplens.org/datasets/movielens/ml-100k/u.item'

ratings_df = pd.read_csv(
    ratings_url, 
    sep='\t', 
    names=['user_id', 'movie_id', 'rating', 'timestamp']
)

movies_df = pd.read_csv(
    movies_url,
    sep='|',
    encoding='latin-1',
    names=['movie_id', 'title', 'release_date', 'video_release', 'imdb_url',
           'unknown', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy',
           'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror',
           'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']
)

print(f"Ratings: {ratings_df.shape[0]:,} rows")
print(f"Movies: {movies_df.shape[0]:,} rows")
movies_df.head()

Ratings: 100,000 rows
Movies: 1,682 rows


,movie_id,title,release_date,video_release,imdb_url,unknown,Action,Adventure,Animation,Children,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Toy%20Story%2...,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
1,2,GoldenEye (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?GoldenEye%20(...,0,1,1,0,0,...,0,0,0,0,0,0,0,1,0,0
2,3,Four Rooms (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Four%20Rooms%...,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,4,Get Shorty (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Get%20Shorty%...,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,Copycat (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Copycat%20(1995),0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0


In [5]:
# Preview ratings data
ratings_df.head()

,user_id,movie_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [6]:
# Preview movies data (with some genre columns)
movies_df[['movie_id', 'title', 'Action', 'Comedy', 'Drama', 'Sci-Fi']].head()

,movie_id,title,Action,Comedy,Drama,Sci-Fi
0,1,Toy Story (1995),0,1,0,0
1,2,GoldenEye (1995),1,0,0,0
2,3,Four Rooms (1995),0,0,0,0
3,4,Get Shorty (1995),1,1,1,0
4,5,Copycat (1995),0,0,1,0


In [7]:
# Dataset statistics
print(f"Total ratings: {len(ratings_df):,}")
print(f"Unique users: {ratings_df['user_id'].nunique():,}")
print(f"Unique movies: {ratings_df['movie_id'].nunique():,}")
print(f"Average rating: {ratings_df['rating'].mean():.2f}")

Total ratings: 100,000
Unique users: 943
Unique movies: 1,682
Average rating: 3.53


---
## Part 2: Collaborative Filtering

**Collaborative Filtering** finds users with similar tastes and recommends what they liked.

Two approaches:
1. **User-Based CF** - Find similar users, recommend their favorites
2. **SVD** - Learn latent factors that explain preferences

### 2.1 User-Based CF with KNN

In [ ]:
# Prepare data for Surprise library
reader = Reader(rating_scale=(1, 5))
surprise_data = Dataset.load_from_df(ratings_df[['user_id', 'movie_id', 'rating']], reader)

trainset, testset = surprise_split(surprise_data, test_size=0.2, random_state=42)
print(f"Training: {trainset.n_ratings:,} | Test: {len(testset):,}")


Training: 80,000 | Test: 20,000


In [9]:
# Build and train User-Based CF model
sim_options = {'name': 'cosine', 'user_based': True}
knn_user = KNNBasic(k=40, sim_options=sim_options, verbose=False)
knn_user.fit(trainset)
print("User-Based CF model trained!")

User-Based CF model trained!


In [10]:
# Evaluate on test set
predictions_knn = knn_user.test(testset)
rmse_knn = accuracy.rmse(predictions_knn, verbose=False)
print(f"User-Based CF RMSE: {rmse_knn:.4f} (lower = better)")

User-Based CF RMSE: 1.0194 (lower = better)


In [11]:
# Sample predictions
print(f"{'User':<8} {'Movie':<8} {'Actual':<10} {'Predicted':<10}")
print("-" * 40)
for pred in predictions_knn[:5]:
    print(f"{pred.uid:<8} {pred.iid:<8} {pred.r_ui:<10.1f} {pred.est:<10.2f}")

User     Movie    Actual     Predicted 
----------------------------------------
907      143      5.0        4.00      
371      210      4.0        4.02      
218      42       4.0        3.92      
829      170      4.0        4.28      
733      277      1.0        3.38      


### 2.2 SVD (Matrix Factorization)

SVD learns hidden features that explain preferences - the technique that won the **Netflix Prize**!

In [12]:
# Build and train SVD model
svd = SVD(n_factors=100, n_epochs=20, random_state=42)
svd.fit(trainset)
print("SVD model trained!")

SVD model trained!


In [13]:
# Evaluate SVD
predictions_svd = svd.test(testset)
rmse_svd = accuracy.rmse(predictions_svd, verbose=False)

print(f"User-Based CF: {rmse_knn:.4f}")
print(f"SVD:           {rmse_svd:.4f}")
print(f"\nSVD is {'better' if rmse_svd < rmse_knn else 'worse'}!")

User-Based CF: 1.0194
SVD:           0.9352

SVD is better!


### 2.3 Getting Top-N Recommendations

In [14]:
def get_top_n_recommendations(model, user_id, n=10, trainset=trainset, movies_df=movies_df):
    """Get top N movie recommendations for a user."""
    all_movie_ids = set(movies_df['movie_id'].unique())
    
    # Get movies the user has already rated
    user_inner_id = trainset.to_inner_uid(user_id)
    rated_movies = set([trainset.to_raw_iid(item) for item, _ in trainset.ur[user_inner_id]])
    
    # Predict ratings for unrated movies
    movies_to_predict = all_movie_ids - rated_movies
    predictions = [(mid, model.predict(user_id, mid).est) for mid in movies_to_predict]
    predictions.sort(key=lambda x: x[1], reverse=True)
    
    # Return top N as DataFrame
    results = []
    for movie_id, pred_rating in predictions[:n]:
        title = movies_df[movies_df['movie_id'] == movie_id]['title'].values[0]
        results.append({'title': title, 'predicted_rating': round(pred_rating, 2)})
    
    return pd.DataFrame(results)

In [15]:
# Get recommendations for User 1
get_top_n_recommendations(svd, user_id=1, n=10)

,title,predicted_rating
0,Dr. Strangelove or: How I Learned to Stop Worr...,4.94
1,One Flew Over the Cuckoo's Nest (1975),4.94
2,Rear Window (1954),4.93
3,Citizen Kane (1941),4.75
4,Schindler's List (1993),4.69
5,"Third Man, The (1949)",4.68
6,To Kill a Mockingbird (1962),4.67
7,"Silence of the Lambs, The (1991)",4.67
8,Casablanca (1942),4.64
9,"Manchurian Candidate, The (1962)",4.55


In [16]:
# What has User 1 rated highly? (For comparison)
user_ratings = ratings_df[ratings_df['user_id'] == 1].merge(movies_df[['movie_id', 'title']])
user_ratings[user_ratings['rating'] == 5][['title', 'rating']].head(10)

,title,rating
5,Groundhog Day (1993),5
6,Delicatessen (1991),5
12,"Pillow Book, The (1995)",5
13,"Horseman on the Roof, The (Hussard sur le toit...",5
17,"Shawshank Redemption, The (1994)",5
19,Star Trek: The Wrath of Khan (1982),5
22,Wallace & Gromit: The Best of Aardman Animatio...,5
28,Breaking the Waves (1996),5
32,Three Colors: Blue (1993),5
33,"Good, The Bad and The Ugly, The (1966)",5


---
## Part 3: Content-Based Filtering

Recommends items similar to what the user liked, based on **item features** (genres).

In [17]:
# Create genre feature matrix
genre_columns = ['Action', 'Adventure', 'Animation', 'Children', 'Comedy',
                 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 
                 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 
                 'Thriller', 'War', 'Western']

genre_matrix = movies_df[genre_columns].values
print(f"Genre matrix: {genre_matrix.shape} (movies x genres)")

Genre matrix: (1682, 18) (movies x genres)


In [18]:
# Calculate movie-to-movie similarity
movie_similarity = cosine_similarity(genre_matrix)
print(f"Similarity matrix: {movie_similarity.shape}")

Similarity matrix: (1682, 1682)


In [19]:
def get_similar_movies(movie_title, n=10):
    """Find movies similar to the given movie based on genres."""
    movie_matches = movies_df[movies_df['title'].str.contains(movie_title, case=False)]
    if len(movie_matches) == 0:
        return print(f"Movie '{movie_title}' not found!")
    
    movie_idx = movie_matches.index[0]
    actual_title = movie_matches.iloc[0]['title']
    sim_scores = movie_similarity[movie_idx]
    similar_indices = sim_scores.argsort()[::-1][1:n+1]
    
    results = []
    for idx in similar_indices:
        results.append({
            'title': movies_df.iloc[idx]['title'],
            'similarity': round(sim_scores[idx], 3),
            'genres': ', '.join([g for g in genre_columns if movies_df.iloc[idx][g] == 1])
        })
    
    print(f"Similar to: {actual_title}")
    return pd.DataFrame(results)

In [20]:
get_similar_movies("Toy Story")

Similar to: Toy Story (1995)


,title,similarity,genres
0,Aladdin and the King of Thieves (1996),1.000,"Animation, Children, Comedy"
1,Aladdin (1992),0.866,"Animation, Children, Comedy, Musical"
2,"Goofy Movie, A (1995)",0.866,"Animation, Children, Comedy, Romance"
3,Jungle2Jungle (1997),0.816,"Children, Comedy"
4,Air Bud (1997),0.816,"Children, Comedy"
5,George of the Jungle (1997),0.816,"Children, Comedy"
6,"Santa Clause, The (1994)",0.816,"Children, Comedy"
7,Little Big League (1994),0.816,"Children, Comedy"
8,Heavyweights (1994),0.816,"Children, Comedy"
9,"Flintstones, The (1994)",0.816,"Children, Comedy"


In [21]:
get_similar_movies("Star Wars")

Similar to: Star Wars (1977)


,title,similarity,genres
0,Return of the Jedi (1983),1.000,"Action, Adventure, Romance, Sci-Fi, War"
1,"Empire Strikes Back, The (1980)",0.913,"Action, Adventure, Drama, Romance, Sci-Fi, War"
2,Starship Troopers (1997),0.894,"Action, Adventure, Sci-Fi, War"
3,"African Queen, The (1951)",0.894,"Action, Adventure, Romance, War"
4,"Last of the Mohicans, The (1992)",0.775,"Action, Romance, War"
5,Time Tracers (1995),0.775,"Action, Adventure, Sci-Fi"
6,Star Trek: The Wrath of Khan (1982),0.775,"Action, Adventure, Sci-Fi"
7,Star Trek III: The Search for Spock (1984),0.775,"Action, Adventure, Sci-Fi"
8,Star Trek IV: The Voyage Home (1986),0.775,"Action, Adventure, Sci-Fi"
9,Star Trek: First Contact (1996),0.775,"Action, Adventure, Sci-Fi"


### Content-Based Recommendations for a User

In [22]:
def get_content_based_recommendations(user_id, n=10):
    """Get content-based recommendations based on user's rating history."""
    user_ratings = ratings_df[ratings_df['user_id'] == user_id]
    liked_movies = user_ratings[user_ratings['rating'] >= 4]['movie_id'].tolist()
    
    if not liked_movies:
        return print(f"User {user_id} hasn't rated any movies 4+ stars!")
    
    # Build user profile from liked movies
    liked_indices = movies_df[movies_df['movie_id'].isin(liked_movies)].index
    user_profile = genre_matrix[liked_indices].mean(axis=0)
    
    # Find similar movies user hasn't seen
    all_similarities = cosine_similarity([user_profile], genre_matrix)[0]
    seen_movies = user_ratings['movie_id'].tolist()
    
    recommendations = []
    for idx, sim in enumerate(all_similarities):
        movie_id = movies_df.iloc[idx]['movie_id']
        if movie_id not in seen_movies:
            recommendations.append((idx, sim))
    
    recommendations.sort(key=lambda x: x[1], reverse=True)
    
    results = []
    for idx, sim in recommendations[:n]:
        results.append({
            'title': movies_df.iloc[idx]['title'],
            'similarity': round(sim, 3),
            'genres': ', '.join([g for g in genre_columns if movies_df.iloc[idx][g] == 1])
        })
    
    return pd.DataFrame(results)

In [23]:
get_content_based_recommendations(user_id=1)

,title,similarity,genres
0,"House of Yes, The (1997)",0.773,"Comedy, Drama, Thriller"
1,Wings of Desire (1987),0.773,"Comedy, Drama, Romance"
2,Manhattan (1979),0.773,"Comedy, Drama, Romance"
3,"American President, The (1995)",0.773,"Comedy, Drama, Romance"
4,"Corrina, Corrina (1994)",0.773,"Comedy, Drama, Romance"
5,Something to Talk About (1995),0.773,"Comedy, Drama, Romance"
6,Don Juan DeMarco (1995),0.773,"Comedy, Drama, Romance"
7,Brassed Off (1996),0.773,"Comedy, Drama, Romance"
8,What Happened Was... (1994),0.773,"Comedy, Drama, Romance"
9,Twelfth Night (1996),0.773,"Comedy, Drama, Romance"


---
## Part 4: Evaluation

**RMSE** tells us how close predictions are to actual ratings. Lower is better.

In [24]:
# Model comparison
print(f"RMSE Results:")
print(f"  SVD:           {rmse_svd:.4f}")
print(f"  User-Based CF: {rmse_knn:.4f}")
print(f"\nSVD predictions are off by ~{rmse_svd:.2f} stars on average")

RMSE Results:
  SVD:           0.9352
  User-Based CF: 1.0194

SVD predictions are off by ~0.94 stars on average


---
## Part 5: The Cold Start Problem

What happens with **new users** (no history) or **new items** (no ratings)?

### Solution 1: Popularity-Based (New Users)

Recommend the most popular movies!

In [25]:
def get_popular_movies(n=10):
    """Get most popular movies (for new users)."""
    movie_stats = ratings_df.groupby('movie_id').agg(
        avg_rating=('rating', 'mean'),
        num_ratings=('rating', 'count')
    ).reset_index()
    
    popular = movie_stats[movie_stats['num_ratings'] >= 50]
    popular = popular.sort_values('avg_rating', ascending=False).head(n)
    popular = popular.merge(movies_df[['movie_id', 'title']])
    
    return popular[['title', 'avg_rating', 'num_ratings']].round(2)

get_popular_movies(10)

,title,avg_rating,num_ratings
0,"Close Shave, A (1995)",4.49,112
1,Schindler's List (1993),4.47,298
2,"Wrong Trousers, The (1993)",4.47,118
3,Casablanca (1942),4.46,243
4,Wallace & Gromit: The Best of Aardman Animatio...,4.45,67
5,"Shawshank Redemption, The (1994)",4.45,283
6,Rear Window (1954),4.39,209
7,"Usual Suspects, The (1995)",4.39,267
8,Star Wars (1977),4.36,583
9,12 Angry Men (1957),4.34,125


### Solution 2: Onboarding Survey

Ask new users what genres they like!

In [26]:
def simulate_onboarding(genre_preferences):
    """Recommend movies based on genre preferences (1-5 scale)."""
    user_profile = np.zeros(len(genre_columns))
    for i, genre in enumerate(genre_columns):
        if genre in genre_preferences:
            user_profile[i] = genre_preferences[genre] / 5.0
    
    similarities = cosine_similarity([user_profile], genre_matrix)[0]
    top_indices = similarities.argsort()[::-1][:10]
    
    results = []
    for idx in top_indices:
        results.append({
            'title': movies_df.iloc[idx]['title'],
            'match': round(similarities[idx], 3),
            'genres': ', '.join([g for g in genre_columns if movies_df.iloc[idx][g] == 1])
        })
    return pd.DataFrame(results)

In [27]:
# New user loves Sci-Fi and Action
simulate_onboarding({'Sci-Fi': 5, 'Action': 5, 'Comedy': 3})

,title,match,genres
0,Timecop (1994),0.921,"Action, Sci-Fi"
1,Barb Wire (1996),0.921,"Action, Sci-Fi"
2,No Escape (1994),0.921,"Action, Sci-Fi"
3,Demolition Man (1993),0.921,"Action, Sci-Fi"
4,Highlander III: The Sorcerer (1994),0.921,"Action, Sci-Fi"
5,"Fifth Element, The (1997)",0.921,"Action, Sci-Fi"
6,Men in Black (1997),0.846,"Action, Adventure, Comedy, Sci-Fi"
7,Mars Attacks! (1996),0.846,"Action, Comedy, Sci-Fi, War"
8,Tank Girl (1995),0.846,"Action, Comedy, Musical, Sci-Fi"
9,Army of Darkness (1993),0.757,"Action, Adventure, Comedy, Horror, Sci-Fi"


### Solution 3: Content-Based (New Items)

Use item features to find users who might like a new movie!

In [28]:
def find_potential_fans(movie_genres, sample_users=100):
    """Find users who might like a new movie based on their genre history."""
    movie_vector = np.zeros(len(genre_columns))
    for i, genre in enumerate(genre_columns):
        if genre in movie_genres:
            movie_vector[i] = 1
    
    user_affinities = []
    for user_id in ratings_df['user_id'].unique()[:sample_users]:
        user_ratings = ratings_df[(ratings_df['user_id'] == user_id) & (ratings_df['rating'] >= 4)]
        if len(user_ratings) == 0:
            continue
        liked_indices = movies_df[movies_df['movie_id'].isin(user_ratings['movie_id'])].index
        if len(liked_indices) == 0:
            continue
        user_profile = genre_matrix[liked_indices].mean(axis=0)
        similarity = cosine_similarity([movie_vector], [user_profile])[0][0]
        user_affinities.append((user_id, similarity))
    
    user_affinities.sort(key=lambda x: x[1], reverse=True)
    return user_affinities[:10]

In [29]:
# New Sci-Fi Action movie - who would like it?
fans = find_potential_fans(['Sci-Fi', 'Action', 'Adventure'])
print("Top users for new Sci-Fi/Action movie:")
for user_id, affinity in fans:
    print(f"  User {user_id}: {affinity:.1%} match")

Top users for new Sci-Fi/Action movie:
  User 127: 96.4% match
  User 102: 78.6% match
  User 254: 78.2% match
  User 8: 77.0% match
  User 22: 71.5% match
  User 20: 68.9% match
  User 200: 68.3% match
  User 28: 67.2% match
  User 290: 65.9% match
  User 97: 65.3% match


---
## Summary

| Approach | How It Works | Best For |
|----------|--------------|----------|
| **Collaborative Filtering** | Find similar users/items | Users with history |
| **Content-Based** | Match item features | New items |
| **Popularity** | Most-liked items | New users |
| **Hybrid** | Combine approaches | Production systems |

Real systems (Netflix, Spotify, Amazon) use **hybrid approaches** combining all techniques!

---
## Optional Challenges

Try these to deepen your understanding:

**Challenge 1: Tune the Models**
- Change `k` in KNNBasic (try 10, 20, 50) - how does RMSE change?
- Change `n_factors` in SVD (try 20, 50, 150) - what's the sweet spot?

**Challenge 2: Find Your Movie Twin**
- Pick a user whose 5-star movies you agree with
- Get recommendations for that user - would you watch them?

**Challenge 3: Genre Detective**
- Use `get_similar_movies()` on your favorite movie
- Do the results make sense? What's missing?

**Challenge 4: Build a Simple Hybrid**
- Average the SVD predicted rating with content-based similarity
- Does combining approaches improve recommendations?

**Challenge 5: Explore the Data**
- Which genre has the highest average rating?
- Which user has rated the most movies?
- What's the most polarizing movie (high variance in ratings)?

In [ ]:
# Your exploration here!